In [1]:
import requests
from bs4 import BeautifulSoup
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
import pandas as pd
import dask.dataframe as dd
import os
from datetime import date, timedelta

page = requests.get("https://www.worldometers.info/coronavirus")

# Soup is a variable containing the HTML of the webpage
soup = BeautifulSoup(page.content, 'lxml')

# Search for the table and extracting it
table = soup.find('table', attrs={'id': 'main_table_countries_yesterday'})
rows = table.find_all("tr", attrs={"style": ""})

data = []
for i, item in enumerate(rows):
    if i == 0:
        data.append(item.text.strip().split("\n")[:16])
    else:
        data.append(item.text.strip().split("\n")[:15])

data[0][13] = data[0][13] + data[0][14]
data[0][14] = 'Population'
data[1].pop()
data[1].insert(0, ' ')

dt = pd.DataFrame(data[1:], columns=data[0][:15]) # Formatting the header
df = dd.from_pandas(dt, npartitions=1)

yesterday = (date.today() - timedelta(days = 1)).strftime(r"%d-%m-%Y")
print(yesterday)
path = "Extracted_data"
if not os.path.exists(path):
    os.makedirs(path)
df.compute().to_csv(path + "\\data_covid19_{}.csv".format(yesterday))

28-04-2022
